# Churn Prediction Model

Name: Nirjhari Joshi

Student ID: IITP_AIML_2506082

Objective: Predict whether a customer will churn within the next 60 days and provide actionable insights for retention teams.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile

zip_path = "d2c churn data package-20260615T085556Z-3-001.zip"

z = zipfile.ZipFile(zip_path)

snapshot = pd.read_csv(
    z.open(
        "d2c churn data package/rfm_modeling_snapshot.csv"
    )
)

print(snapshot.shape)
snapshot.head()

(2400, 29)


,customer_id,snapshot_date,city_tier,age_group,acquisition_channel,loyalty_tier,preferred_category,marketing_consent,recency_days,frequency_180d,...,sessions_30d,product_views_30d,cart_adds_30d,wishlist_adds_30d,abandoned_carts_30d,email_opens_30d,campaign_clicks_30d,last_visit_days_ago,churn_next_60d,split
0,CUST00001,2025-09-30,Tier 1,18-24,Instagram,Silver,Makeup,Yes,107,1,...,1,4,0,0,0,2,0,20,1,train
1,CUST00002,2025-09-30,Tier 2,25-34,Marketplace,Silver,Hair Care,Yes,40,1,...,8,31,4,2,3,0,0,0,0,train
2,CUST00003,2025-09-30,Tier 1,25-34,Influencer,NaN,Skin Care,Yes,171,1,...,1,3,0,0,0,0,0,26,1,train
3,CUST00004,2025-09-30,Tier 3,25-34,Google Search,NaN,Fragrance,No,131,1,...,1,6,0,0,0,0,0,14,1,train
4,CUST00005,2025-09-30,Tier 3,35-44,Organic,Gold,Hair Care,Yes,38,3,...,18,95,4,1,1,3,1,9,0,train


## Leakage Check

The target variable is `churn_next_60d`.

Only features available at or before the snapshot date are used.

The following fields are excluded from modeling:

* customer_id
* snapshot_date
* split
* churn_next_60d (target)

No future information is used as an input feature.


In [2]:
from sklearn.preprocessing import LabelEncoder

df = snapshot.copy()

target = 'churn_next_60d'

drop_cols = [
    'customer_id',
    'snapshot_date',
    'split',
    target
]

X = df.drop(columns=drop_cols)

y = df[target]

for col in X.select_dtypes(include='object').columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

print(X.shape)

(2400, 25)


In [3]:
train_df = df[df['split'] == 'train']
valid_df = df[df['split'] == 'validation']
test_df  = df[df['split'] == 'test']

print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)

(1728, 29)
(336, 29)
(336, 29)


In [4]:
from sklearn.preprocessing import LabelEncoder

target = 'churn_next_60d'

drop_cols = [
    'customer_id',
    'snapshot_date',
    'split',
    target
]

train_X = train_df.drop(columns=drop_cols).copy()
valid_X = valid_df.drop(columns=drop_cols).copy()
test_X  = test_df.drop(columns=drop_cols).copy()

train_y = train_df[target]
valid_y = valid_df[target]
test_y  = test_df[target]

for col in train_X.select_dtypes(include='object').columns:

    le = LabelEncoder()

    combined = pd.concat([
        train_X[col],
        valid_X[col],
        test_X[col]
    ])

    le.fit(combined.astype(str))

    train_X[col] = le.transform(train_X[col].astype(str))
    valid_X[col] = le.transform(valid_X[col].astype(str))
    test_X[col] = le.transform(test_X[col].astype(str))

print(train_X.shape)
print(valid_X.shape)
print(test_X.shape)

(1728, 25)
(336, 25)
(336, 25)


In [5]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

baseline_model.fit(
    train_X,
    train_y
)

print("Baseline model trained")

Baseline model trained


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [6]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    random_state=42
)

rf_model.fit(
    train_X,
    train_y
)

print("Random Forest trained")

Random Forest trained


In [7]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def evaluate_model(model, X, y):

    pred = model.predict(X)

    proba = model.predict_proba(X)[:,1]

    return {
        "Accuracy": accuracy_score(y,pred),
        "Precision": precision_score(y,pred),
        "Recall": recall_score(y,pred),
        "F1": f1_score(y,pred),
        "ROC_AUC": roc_auc_score(y,proba)
    }

baseline_metrics = evaluate_model(
    baseline_model,
    valid_X,
    valid_y
)

rf_metrics = evaluate_model(
    rf_model,
    valid_X,
    valid_y
)

print("Logistic Regression")
print(baseline_metrics)

print("\nRandom Forest")
print(rf_metrics)

Logistic Regression
{'Accuracy': 0.8154761904761905, 'Precision': 0.8195488721804511, 'Recall': 0.7414965986394558, 'F1': 0.7785714285714286, 'ROC_AUC': np.float64(0.8842097685635101)}

Random Forest
{'Accuracy': 0.8035714285714286, 'Precision': 0.8, 'Recall': 0.7346938775510204, 'F1': 0.7659574468085106, 'ROC_AUC': np.float64(0.8748875211460245)}


In [8]:
test_pred = baseline_model.predict(test_X)

test_proba = baseline_model.predict_proba(test_X)[:,1]

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

test_metrics = {
    "Accuracy": accuracy_score(test_y,test_pred),
    "Precision": precision_score(test_y,test_pred),
    "Recall": recall_score(test_y,test_pred),
    "F1": f1_score(test_y,test_pred),
    "ROC_AUC": roc_auc_score(test_y,test_proba)
}

print(test_metrics)

cm = confusion_matrix(test_y,test_pred)

print("\nConfusion Matrix")
print(cm)

{'Accuracy': 0.8214285714285714, 'Precision': 0.8333333333333334, 'Recall': 0.8035714285714286, 'F1': 0.8181818181818182, 'ROC_AUC': np.float64(0.8847434807256236)}

Confusion Matrix
[[141  27]
 [ 33 135]]


In [9]:
feature_importance = pd.DataFrame({
    'Feature': train_X.columns,
    'Coefficient':
    abs(
        baseline_model.coef_[0]
    )
})

feature_importance = feature_importance.sort_values(
    'Coefficient',
    ascending=False
)

feature_importance.head(10)

,Feature,Coefficient
9,return_rate_180d,1.620335
14,negative_ticket_rate_90d,0.917297
10,avg_discount_pct_180d,0.834385
13,ticket_count_90d,0.662011
5,marketing_consent,0.217325
12,category_diversity_180d,0.212373
7,frequency_180d,0.142139
23,campaign_clicks_30d,0.126199
11,avg_rating_180d,0.089095
21,abandoned_carts_30d,0.060659


In [10]:
import joblib

joblib.dump(
    baseline_model,
    'model.pkl'
)

print("Model saved")

Model saved


In [11]:
import json

metrics = {
    "accuracy": 0.8214,
    "precision": 0.8333,
    "recall": 0.8036,
    "f1_score": 0.8182,
    "roc_auc": 0.8847,
    "threshold": 0.50,
    "true_negative": 141,
    "false_positive": 27,
    "false_negative": 33,
    "true_positive": 135
}

with open(
    'metrics.json',
    'w'
) as f:
    json.dump(
        metrics,
        f,
        indent=4
    )

print("metrics.json created")

metrics.json created
